In [ ]:
from dotenv import load_dotenv
from typing import Any
import os
import litellm_practice.litellm_practice as litellm_practice

In [ ]:
class LLM:
    def __init__(self):
        env = os.getenv('ENV','local')
        load_dotenv(f'.env.{env}',override=True)
        self.api_key = os.getenv("API_KEY").strip()
        self.model = os.getenv("MODEL").strip()
        print(f"=========={self.model}")
        self.url = os.getenv("BASE_URL").strip()
        try:
            self.thinking_timeout = float(os.getenv("TINKING_TIMEOUT",30).strip())
            self.num_retries = int(os.getenv("RETRY_NUM",2).strip())
            self.temperature = int(os.getenv("TEMPERATURE",1).strip())
            self.stream = True if os.getenv("STREAM").strip() == 'True' else False
            self.max_token = int(os.getenv("MAX_TOKEN").strip()) if os.getenv("MAX_TOKEN") else None
        except Exception as e:
            raise Exception(f'配置文件传入非法参数。错误信息：{e}')
        
    async def run(self,message: list[dict],**kwargs: Any):
        param : dict[str,Any] = {
            'model': self.model,
            'api_key':self.api_key,
            'timeout':self.thinking_timeout, 
            'messages':message,
            'temperature': self.temperature,
            'stream': self.stream,
            'base_url': self.url,
            'max_tokens':self.max_token,
            'num_retries':self.num_retries,
            **kwargs
        }
        print(f"model: {param['model']}")
        try:
            res = await litellm_practice.acompletion(**param)
        except Exception as e:
            raise Exception(f"模型{self.model}调用失败，请检查配置。错误信息：{e}")
        return res
    async def send_message_once(self,message:str):
        m = []
        mes :dict[str,Any] = {
            'role': 'user',
            'content': message
        }
        m.append(mes)
        res = await self.run(m)
        return res

In [77]:
llm = LLM()
res = await llm.send_message_once('你是什么模型')
print(res)

==========deepseek/deepseek-v4-flash
model: deepseek/deepseek-v4-flash
ModelResponse(id='34779909-2804-4e15-9070-50e8b15dba08', created=1784617524, model='deepseek-v4-flash', object='chat.completion', system_fingerprint='fp_8b330d02d0_prod0820_fp8_kvcache_20260402', choices=[Choices(finish_reason='stop', index=0, message=Message(content='我是DeepSeek最新版本的模型，由深度求索公司创造。具体来说：\n\n- **模型类型**：我是一个纯文本对话模型，支持多种交互方式\n- **知识截止日期**：2025年5月\n- **核心能力**：支持1M超长上下文（可以一次性处理像《三体》三部曲这样的长篇内容）、联网搜索（需手动开启）、文件上传处理（图片、PDF、Word、Excel、PPT等）\n\n我是免费的，没有任何收费计划，你可以尽情使用！如果你想了解更具体的版本号或技术细节，建议查阅DeepSeek官方文档和公告。\n\n有什么我可以帮你的吗？😊', role='assistant', tool_calls=None, function_call=None, reasoning_content='嗯，用户问“你是什么模型”，这是一个简单的身份询问。用户可能刚接触我，想了解我的基本信息和能力范围。需要清晰告知我是DeepSeek，并说明主要版本和核心特点。想到了可以简要列出几个关键点：免费、上下文长度、文件处理、联网搜索等，这样能让用户快速知道我有什么功能。最后可以表达乐意帮助的友好态度，让对话更自然。', provider_specific_fields=None), provider_specific_fields={})], usage=Usage(completion_tokens=223, prompt_tokens=7, total_tokens=230, completion_tokens_details=Compl